---
# `Normalization Layer and Feed Forward Layer`
---

- It is step 6
- Data is passed through these layers.
- By using this addition we came up with proper contextual embeddings of words in the sentence in case of Encoder
- So let us say, we are having input - Hey, how are you  --> passed to Encoder to generate Numeric embeddings --> Contextual Numeric embeddings

---
---
# `Detailed Notes 1`
---
---

# Layer Normalization and Feed Forward Network (FFN) in Transformers

After the **Multi-Head Attention** layer, the Transformer performs two additional operations:

1. **Layer Normalization (LayerNorm)**
2. **Feed Forward Network (FFN)**

These two components are repeated in **every Transformer block**.

---

# Transformer Block Architecture

```text
                 Input
                   │
                   ▼
        Multi-Head Attention
                   │
        + Residual Connection
                   │
                   ▼
          Layer Normalization
                   │
                   ▼
       Feed Forward Network
                   │
        + Residual Connection
                   │
                   ▼
          Layer Normalization
                   │
                   ▼
                Output
```

> **Note:** The original Transformer ("Attention Is All You Need") applies **Residual → LayerNorm** ("Post-LN"). Many modern LLMs (GPT-2, LLaMA, Gemma, etc.) use **LayerNorm → Module → Residual** ("Pre-LN") because it improves training stability. The purpose of LayerNorm is the same in both designs.

---

# What is Layer Normalization?

**Layer Normalization** is a technique used to **normalize the activations of each token across its feature dimensions**.

Its purpose is to:

* Stabilize training
* Speed up convergence
* Prevent activations from becoming too large or too small
* Improve gradient flow in deep networks

---

# Why Do We Need Layer Normalization?

Suppose after Multi-Head Attention we obtain:

```text
Token 1

[100, 250, -80, 420]

Token 2

[-300, 600, 120, -50]
```

The values vary significantly.

Large differences can make optimization unstable.

Layer Normalization rescales these values to a more consistent distribution.

Example:

```text
Before

[100,250,-80,420]

↓

After LayerNorm

[-0.32,0.55,-1.40,1.17]
```

The relative information is preserved while the scale becomes more manageable.

---

# Intuition

Think of students taking exams.

Class A

```text
Marks

20

30

95

100
```

Class B

```text
Marks

45

55

60

70
```

Comparing raw scores is difficult.

Normalization puts them on a comparable scale.

Similarly, LayerNorm ensures that activations have a consistent scale.

---

# How Layer Normalization Works

Suppose one token has four features.

```text
[5,10,15,20]
```

---

### Step 1: Compute Mean

[
$\mu=\frac{5+10+15+20}{4}$
]

[
\mu=12.5
]

---

### Step 2: Compute Variance

Measure how spread out the values are.

[
$\sigma^2=\frac{1}{n}\sum(x_i-\mu)^2$
]

---

### Step 3: Normalize

[
$\hat{x}=\frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}$
]

where:

* $(\epsilon)$ is a very small constant added to avoid division by zero.

---

### Step 4: Learn Scale and Shift

LayerNorm includes trainable parameters:

* **γ (gamma)** → scaling
* **β (beta)** → shifting

Final output:

[
$y=\gamma\hat{x}+\beta$
]

This allows the model to learn the most useful scale for each feature.

---

# LayerNorm Formula

[
$\boxed{
y=\gamma
\left(
\frac{x-\mu}
{\sqrt{\sigma^2+\epsilon}}
\right)
+\beta
}$
]

---

# Visual Representation

```text
Input

↓

Compute Mean

↓

Compute Variance

↓

Normalize

↓

Scale (γ)

↓

Shift (β)

↓

Output
```

---

# Why Not Batch Normalization?

Batch Normalization works well for CNNs but has limitations for Transformers.

| Batch Normalization                         | Layer Normalization                  |
| ------------------------------------------- | ------------------------------------ |
| Normalizes across the batch                 | Normalizes each token independently  |
| Depends on batch size                       | Independent of batch size            |
| Less suitable for variable-length sequences | Well suited for NLP and Transformers |
| Common in CNNs                              | Standard in Transformers             |

Because Transformers often process variable-length sequences and use different batch sizes, **LayerNorm is preferred**.

---

# Feed Forward Network (FFN)

## What is FFN?

The **Feed Forward Network** is a small fully connected neural network applied **independently to each token** after the attention layer.

Attention allows tokens to exchange information.

The FFN then transforms each token's representation into a richer feature representation.

---

# Why Do We Need FFN?

Multi-Head Attention mixes information between tokens.

However, we still need a mechanism to perform **non-linear feature transformation** on each token individually.

The FFN provides this capability.

---

# Architecture

```text
Input

↓

Linear

↓

Activation

↓

Linear

↓

Output
```

Every token passes through the same FFN using the same weights.

---

# Mathematical Formula

The original Transformer uses:

[
$\boxed{
FFN(x)=
\max(0,xW_1+b_1)W_2+b_2
}$
]

where:

* First Linear Layer projects to a higher dimension.
* ReLU is the activation in the original paper.
* Second Linear Layer projects back to the model dimension.

> Many modern LLMs replace **ReLU** with **GELU** (e.g., BERT, GPT-2) or **SwiGLU** (e.g., LLaMA family) because they often improve performance.

---

# Example

Suppose

```text
Embedding Size

512
```

The FFN expands it to:

```text
2048
```

then projects it back:

```text
512
```

```text
512

↓

Linear

↓

2048

↓

Activation

↓

Linear

↓

512
```

This expansion gives the network more capacity to learn complex patterns.

---

# Workflow

```text
Attention Output

↓

Linear Layer

↓

Activation

↓

Linear Layer

↓

Output
```

---

# Example

Suppose the attention output is:

```text
[0.2,0.4,0.6]
```

After the first linear layer:

```text
[0.9,1.2,-0.5,0.3]
```

After activation:

```text
[0.9,1.2,0.0,0.3]
```

(assuming ReLU)

After the second linear layer:

```text
[0.3,0.7,0.4]
```

This becomes the transformed representation for that token.

---

# Why Increase the Dimension?

Instead of

```text
512

↓

512
```

we use

```text
512

↓

2048

↓

512
```

The intermediate higher-dimensional space allows the model to learn richer, more expressive features before compressing them back.

---

# TensorFlow/Keras Implementation

### Layer Normalization

```python
from tensorflow.keras.layers import LayerNormalization

layer_norm = LayerNormalization(epsilon=1e-6)
```

---

### Feed Forward Network

```python
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

ffn = Sequential([
    Dense(2048, activation="gelu"),
    Dense(512)
])
```

---

# PyTorch Implementation

### Layer Normalization

```python
import torch.nn as nn

layer_norm = nn.LayerNorm(512)
```

---

### Feed Forward Network

```python
import torch.nn as nn

ffn = nn.Sequential(
    nn.Linear(512, 2048),
    nn.GELU(),
    nn.Linear(2048, 512)
)
```

---

# Complete Transformer Block

```text
Input Embeddings

        │

        ▼

Multi-Head Attention

        │

Residual Connection

        │

Layer Normalization

        │

Feed Forward Network

        │

Residual Connection

        │

Layer Normalization

        │

Output
```

---

# Multi-Head Attention vs Feed Forward Network

| Feature             | Multi-Head Attention                | Feed Forward Network                         |
| ------------------- | ----------------------------------- | -------------------------------------------- |
| Purpose             | Exchange information between tokens | Transform each token independently           |
| Uses Q, K, V        | Yes                                 | No                                           |
| Captures Context    | Yes                                 | No                                           |
| Processes Tokens    | Together                            | Independently (same network for every token) |
| Contains Activation | No (attention itself)               | Yes (ReLU, GELU, SwiGLU, etc.)               |

---

# Layer Normalization vs Feed Forward Network

| Feature                  | Layer Normalization   | Feed Forward Network         |
| ------------------------ | --------------------- | ---------------------------- |
| Purpose                  | Stabilize activations | Learn richer representations |
| Learnable Parameters     | γ and β               | Weights and biases           |
| Uses Activation Function | No                    | Yes                          |
| Changes Feature Values   | Normalizes them       | Learns new transformations   |
| Applied Per Token        | Yes                   | Yes                          |

---

# Complete Transformer Flow

```text
Input Text

↓

Tokenizer

↓

Embeddings

↓

Positional Encoding

↓

Multi-Head Attention

↓

Residual Connection

↓

Layer Normalization

↓

Feed Forward Network

↓

Residual Connection

↓

Layer Normalization

↓

Output

↓

Next Transformer Block
```

---

# Interview Summary

> After **Multi-Head Attention**, the Transformer uses **Layer Normalization** to stabilize training by normalizing each token's feature vector. A **Residual Connection** helps preserve information and improves gradient flow. The output is then passed through a **Feed Forward Network (FFN)**, which consists of two linear layers separated by a non-linear activation such as GELU. The FFN is applied independently to every token and learns richer feature representations. Finally, another residual connection and layer normalization are applied before passing the output to the next Transformer block. Together, attention enables communication between tokens, while the FFN enhances each token's representation individually.
